In [1]:
import numpy as np

from numba import njit

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

import optuna
from optuna.visualization import plot_pareto_front
from optuna.trial import TrialState

from dysts.maps import Henon

import time

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


# Init

## Init Funcs

In [10]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scatter(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scatter(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [11]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [12]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [13]:
def weight_plot(weights, dims):
    dim_names = ["x", "y", "z"][:dims]
    num_features_per_type = len(weights) // 2
    num_nodes = num_features_per_type // dims

    labels = []
    for n in range(num_nodes):
        for d in range(dims):
            labels.append(f"Node {n + 1} {dim_names[d]} Pos")
            labels.append(f"Node {n + 1} {dim_names[d]} Vel")

    weights_flat = weights.flatten()
    pos_part = weights_flat[:num_features_per_type]
    vel_part = weights_flat[num_features_per_type:]

    combined_weights = np.empty_like(weights_flat)
    combined_weights[0::2] = pos_part
    combined_weights[1::2] = vel_part

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=combined_weights,
                marker_color=np.where(combined_weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Node State & Dimension",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [14]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    wall_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"
    if wall_nodes is not None and wall_nodes[0] != -1:
        node_colors[wall_nodes] = "blue"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

## Calc Init

In [15]:
@njit(cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))
    disp_reshaped = disp.reshape(num_nodes, dims)

    for i in range(len(connections_list)):
        idx_a = connections_list[i, 0]
        idx_b = connections_list[i, 1]

        delta = np.zeros(dims)
        dist_sq = 0.0
        for j in range(dims):
            pos_a = initial_pos[idx_a, j] + disp_reshaped[idx_a, j]
            pos_b = initial_pos[idx_b, j] + disp_reshaped[idx_b, j]
            delta[j] = pos_b - pos_a
            dist_sq += delta[j] ** 2

        dist = np.sqrt(dist_sq)

        mag = k_vals[i] * (dist - rest_lens[i])

        for j in range(dims):
            f_component = mag * (delta[j] / dist)
            forces[idx_a, j] += f_component
            forces[idx_b, j] -= f_component

    return forces.reshape(-1)

In [16]:
@njit(cache=True)
def run_simulation(
    steps, dt, m_inv_diag, c_diag, U, initial_pos, connections_list, k_vals, rest_lens, wall_nodes=[-1]
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2
        
        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + .5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Simple Spring MC

In [9]:
@njit
def compute_states(inputs, res_size, W_in, W_res, bias, alpha):
    steps = len(inputs)
    X_data = np.zeros((steps + 1, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = inputs @ W_in.T

    for i in range(1, steps + 1):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data[1:]

In [10]:
def evaluate_capacity(X_train, X_test, Y_train, Y_test, num_lags):
    model = Ridge(alpha=1e-5)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    c_k = np.zeros(num_lags)
    for k in range(num_lags):
        y_true = Y_test[:, k]
        y_hat = Y_pred[:, k]

        cov_matrix = np.cov(y_true, y_hat)
        cov = cov_matrix[0, 1]
        var_true = cov_matrix[0, 0]
        var_pred = cov_matrix[1, 1]

        denom = var_true * var_pred
        c_k[k] = (cov**2) / denom if denom > 1e-12 else 0.0

    return c_k

In [11]:
def trial(input, X_states, k_max, total_steps, start_idx, train_steps, test_steps):
    u = input
    X = X_states

    X_clean = X[start_idx:]
    X_train = X_clean[:train_steps]
    X_test = X_clean[train_steps : train_steps + test_steps]

    Y_deg1 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg1[k:, k - 1] = np.sqrt(3.0) * u_delayed
    Y_deg2 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg2[k:, k - 1] = np.sqrt(5.0) * (3.0 * (u_delayed**2) - 1.0) / 2.0

    Y_deg3 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg3[k:, k - 1] = (
            np.sqrt(7.0) * (5.0 * (u_delayed**3) - 3.0 * u_delayed) / 2.0
        )

    # Degree 1
    Y1_clean = Y_deg1[start_idx:]
    Y1_train = Y1_clean[:train_steps]
    Y1_test = Y1_clean[train_steps : train_steps + test_steps]
    c_deg1 = evaluate_capacity(X_train, X_test, Y1_train, Y1_test, k_max)
    # Degree 2
    Y2_clean = Y_deg2[start_idx:]
    Y2_train = Y2_clean[:train_steps]
    Y2_test = Y2_clean[train_steps : train_steps + test_steps]
    c_deg2 = evaluate_capacity(X_train, X_test, Y2_train, Y2_test, k_max)
    # Degree 3
    Y3_clean = Y_deg3[start_idx:]
    Y3_train = Y3_clean[:train_steps]
    Y3_test = Y3_clean[train_steps : train_steps + test_steps]
    c_deg3 = evaluate_capacity(X_train, X_test, Y3_train, Y3_test, k_max)

    k_cross = 25
    cross_pairs = [
        (k1, k2) for k1 in range(1, k_cross + 1) for k2 in range(k1 + 1, k_cross + 1)
    ]
    num_cross_pairs = len(cross_pairs)
    Y_cross2 = np.zeros((total_steps, num_cross_pairs))
    for idx, (k1, k2) in enumerate(cross_pairs):
        max_k = k2
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]
        Y_cross2[max_k:, idx] = 3.0 * (u_k1 * u_k2)

    # Cross Degree 2
    Y_cross2_clean = Y_cross2[start_idx:]
    c_cross2 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross2_clean[:train_steps],
        Y_cross2_clean[train_steps : train_steps + test_steps],
        num_cross_pairs,
    )

    k_cross3 = 25
    # Cross Degree 3 (2+1 interaction)
    pairs_21 = [
        (k1, k2)
        for k1 in range(1, k_cross3 + 1)
        for k2 in range(1, k_cross3 + 1)
        if k1 != k2
    ]
    num_21 = len(pairs_21)
    Y_cross3_21 = np.zeros((total_steps, num_21))
    for idx, (k1, k2) in enumerate(pairs_21):
        max_k = max(k1, k2)
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]

        # sqrt(5) * P2 * sqrt(3) * P1 = sqrt(15) * ((3*u^2 - 1)/2) * u
        p2 = np.sqrt(5.0) * (3.0 * (u_k1**2) - 1.0) / 2.0
        p1 = np.sqrt(3.0) * u_k2
        Y_cross3_21[max_k:, idx] = p2 * p1
    # Cross Degree 3 (1+1+1 interaction)
    triples_111 = [
        (k1, k2, k3)
        for k1 in range(1, k_cross3 + 1)
        for k2 in range(k1 + 1, k_cross3 + 1)
        for k3 in range(k2 + 1, k_cross3 + 1)
    ]
    num_111 = len(triples_111)
    Y_cross3_111 = np.zeros((total_steps, num_111))
    for idx, (k1, k2, k3) in enumerate(triples_111):
        max_k = k3
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]
        u_k3 = u[max_k - k3 : len(u) - k3, 0]

        Y_cross3_111[max_k:, idx] = (3.0 * np.sqrt(3.0)) * (u_k1 * u_k2 * u_k3)
    # Cross Degree 3 (2+1 interaction)
    c_cross3_21 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross3_21[start_idx : start_idx + train_steps],
        Y_cross3_21[start_idx + train_steps : start_idx + train_steps + test_steps],
        num_21,
    )
    # Cross Degree 3 (1+1+1 interaction)
    c_cross3_111 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross3_111[start_idx : start_idx + train_steps],
        Y_cross3_111[start_idx + train_steps : start_idx + train_steps + test_steps],
        num_111,
    )

    return [
        c_deg1,
        c_deg2,
        c_deg3,
        c_cross2,
        c_cross3_21,
        c_cross3_111,
    ]

In [12]:
rng_vals = [0, 42, 123, 456, 789, 999]

N = 30
dist_between = 2.5
dist_y = 1.0
x = np.zeros(N * 3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between * i
y = np.tile(np.array([0, 1 * dist_y, 2 * dist_y]), N)
nodes_pos = np.column_stack((x, y))

sigma = 0
dt = 0.01
input_force = 10
m_val = 0.01
c_val = 0.2
k_wall_val = 5
k_between_val = 4
target_node_count = 10

N_step = int(N / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3
starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()
num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_ids = np.arange(x.size)
starts = 3 * np.arange(N)
connection_src = np.column_stack([starts, starts + 1]).flatten()
connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()
betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]
src_nodes = np.concatenate([connection_src, between_src])
dst_nodes = np.concatenate([connection_dst, between_dst])
connections_list = np.column_stack((src_nodes, dst_nodes))

mu = lambda x: np.log(x) - (sigma**2 / 2)

rng_trials_results = np.zeros((len(rng_vals), 6))

for i, rng_val in enumerate(rng_vals):
    rng = np.random.default_rng(rng_val)
    k_max = int(2 * 100)
    transient = 1000
    train_steps = 5000
    test_steps = 3000
    start_idx = max(transient, k_max)
    total_steps = start_idx + train_steps + test_steps
    u = rng.uniform(-1.0, 1.0, size=(total_steps, 1))
    U = np.zeros((total_steps, matrix_size))
    for j, node_index in enumerate(target_nodes):
        U[::, node_index * dims] = u.reshape(-1)
    U = U * input_force

    m_nodes = rng.lognormal(mean=mu(m_val), sigma=sigma, size=num_nodes)
    m_nodes = np.clip(m_nodes, a_min=0.001, a_max=None)
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = rng.lognormal(mean=mu(c_val), sigma=sigma, size=num_nodes)
    c_nodes = np.clip(c_nodes, a_min=0.001, a_max=None)
    c_diag = np.repeat(c_nodes, dims)

    k_connection_vals = rng.lognormal(
        mean=mu(k_wall_val), sigma=sigma, size=connection_src.shape[0] // 2
    ).repeat(2)
    k_between_vals = rng.lognormal(
        mean=mu(k_between_val), sigma=sigma, size=between_src.shape[0]
    )
    k_vals = np.concatenate([k_connection_vals, k_between_vals])

    rest_connection_lens = rng.lognormal(
        mean=mu(dist_y), sigma=sigma, size=connection_src.shape[0] // 2
    ).repeat(2)
    rest_between_lens = rng.lognormal(
        mean=mu(dist_between), sigma=sigma, size=between_src.shape[0]
    )
    rest_lens = np.concatenate([rest_connection_lens, rest_between_lens])

    displacement, velocity = run_simulation(
        steps=total_steps,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        rest_lens=rest_lens,
        wall_nodes=wall_nodes,
    )

    movement_nodes = 1 + np.arange(N) * 3
    movement_idx = movement_nodes * dims
    X = np.column_stack((displacement[:, movement_idx], velocity[:, movement_idx]))

    values = trial(u, X, k_max, total_steps, start_idx, train_steps, test_steps)
    values_sum = np.array([np.sum(c) for c in values])
    rng_trials_results[i] = values_sum

rng_trials_results_avg = rng_trials_results.sum(axis=0) / len(rng_vals)

In [13]:
categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

fig_bar = go.Figure(
    data=go.Bar(
        x=categories,
        y=rng_trials_results_avg,
        text=[f"{v:.2f}" for v in rng_trials_results_avg],
        textposition="auto",
    )
)

fig_bar.update_layout(
    title=f"Total Capacity: {sum(rng_trials_results_avg):.2f}",
    xaxis_title="Capacity Component",
    yaxis_title="Total Capacity (MC)",
    template="plotly_white",
    width=750,
    height=450,
)

fig_bar.show()

In [17]:
rng_vals = [0, 42, 123, 456, 789, 999]

N = 30
dist_between = 2.5
dist_y = 1.0
x = np.zeros(N * 3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between * i
y = np.tile(np.array([0, 1 * dist_y, 2 * dist_y]), N)
nodes_pos = np.column_stack((x, y))

sigma = .5
dt = 0.01
input_force = 10
m_val = 0.01
c_val = 0.2
k_val = 5
k_between_val = 4
target_node_count = 10

N_step = int(N / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3
starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()
num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_ids = np.arange(x.size)
starts = 3 * np.arange(N)
connection_src = np.column_stack([starts, starts + 1]).flatten()
connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()
betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]
src_nodes = np.concatenate([connection_src, between_src])
dst_nodes = np.concatenate([connection_dst, between_dst])
connections_list = np.column_stack((src_nodes, dst_nodes))

mu = lambda x: np.log(x) - (sigma**2 / 2)

rng_trials_results = np.zeros((len(rng_vals), 6))

for i, rng_val in enumerate(rng_vals):
    rng = np.random.default_rng(rng_val)
    k_max = int(2 * 100)
    transient = 1000
    train_steps = 5000
    test_steps = 3000
    start_idx = max(transient, k_max)
    total_steps = start_idx + train_steps + test_steps
    u = rng.uniform(-1.0, 1.0, size=(total_steps, 1))
    U = np.zeros((total_steps, matrix_size))
    for j, node_index in enumerate(target_nodes):
        U[::, node_index * dims] = u.reshape(-1)
    U = U * input_force

    m_nodes = rng.lognormal(mean=mu(m_val), sigma=sigma, size=num_nodes)
    # m_nodes = np.clip(m_nodes, a_min=0.001, a_max=None)
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = rng.lognormal(mean=mu(c_val), sigma=sigma, size=num_nodes)
    # c_nodes = np.clip(c_nodes, a_min=0.001, a_max=None)
    c_diag = np.repeat(c_nodes, dims)

    k_connection_vals = rng.lognormal(
        mean=mu(k_val), sigma=sigma, size=connection_src.shape[0] // 2
    ).repeat(2)
    k_between_vals = rng.lognormal(
        mean=mu(k_between_val), sigma=sigma, size=between_src.shape[0]
    )
    k_vals = np.concatenate([k_connection_vals, k_between_vals])

    rest_connection_lens = rng.lognormal(
        mean=mu(dist_y), sigma=sigma, size=connection_src.shape[0] // 2
    ).repeat(2)
    rest_between_lens = rng.lognormal(
        mean=mu(dist_between), sigma=sigma, size=between_src.shape[0]
    )
    rest_lens = np.concatenate([rest_connection_lens, rest_between_lens])

    displacement, velocity = run_simulation(
        steps=total_steps,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        rest_lens=rest_lens,
        wall_nodes=wall_nodes,
    )

    movement_nodes = 1 + np.arange(N) * 3
    movement_idx = movement_nodes * dims
    X = np.column_stack((displacement[:, movement_idx], velocity[:, movement_idx]))

    values = trial(u, X, k_max, total_steps, start_idx, train_steps, test_steps)
    values_sum = np.array([np.sum(c) for c in values])
    rng_trials_results[i] = values_sum

rng_trials_results_avg = rng_trials_results.sum(axis=0) / len(rng_vals)

In [18]:
categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

fig_bar = go.Figure(
    data=go.Bar(
        x=categories,
        y=rng_trials_results_avg,
        text=[f"{v:.2f}" for v in rng_trials_results_avg],
        textposition="auto",
    )
)

fig_bar.update_layout(
    title=f"Total Capacity: {sum(rng_trials_results_avg):.2f}",
    xaxis_title="Capacity Component",
    yaxis_title="Total Capacity (MC)",
    template="plotly_white",
    width=750,
    height=450,
)

fig_bar.show()

# Bayesian

In [17]:
N = 30
dist_between = 2.5
dist_y = 1.0
x = np.zeros(N * 3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between * i
y = np.tile(np.array([0, 1 * dist_y, 2 * dist_y]), N)
nodes_pos = np.column_stack((x, y))

In [18]:
dt = 0.01
target_node_count = 10

mu = lambda x, sigma: np.log(x) - (sigma**2 / 2)

k_max = int(2 * 100)
transient = 1000
train_steps = 5000
test_steps = 3000
start_idx = max(transient, k_max)
total_steps = start_idx + train_steps + test_steps

In [19]:
N_step = int(N / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3
starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()
num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

In [20]:
node_ids = np.arange(x.size)
starts = 3 * np.arange(N)
connection_src = np.column_stack([starts, starts + 1]).flatten()
connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()
betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]
src_nodes = np.concatenate([connection_src, between_src])
dst_nodes = np.concatenate([connection_dst, between_dst])
connections_list = np.column_stack((src_nodes, dst_nodes))

In [21]:
def evaluate_capacity(X_train, X_test, Y_train, Y_test, num_lags, ridge_alpha):
    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    c_k = np.zeros(num_lags)
    for k in range(num_lags):
        y_true = Y_test[:, k]
        y_hat = Y_pred[:, k]

        cov_matrix = np.cov(y_true, y_hat)
        cov = cov_matrix[0, 1]
        var_true = cov_matrix[0, 0]
        var_pred = cov_matrix[1, 1]

        denom = var_true * var_pred
        c_k[k] = (cov**2) / denom if denom > 1e-12 else 0.0

    return c_k

In [22]:
def spring_trial(
    rng_seed,
    input_force,
    m_val,
    m_spread,
    c_val,
    c_spread,
    k_wall_val,
    k_wall_spread,
    k_between_val,
    k_between_spread,
    wall_rest_val,
    wall_rest_spread,
    between_rest_val,
    between_rest_spread,
):
    rng = np.random.default_rng(rng_seed)

    u = rng.uniform(-1.0, 1.0, size=(total_steps, 1))
    U = np.zeros((total_steps, matrix_size))
    for j, node_index in enumerate(target_nodes):
        U[::, node_index * dims] = u.reshape(-1)
    U = U * input_force

    m_nodes = rng.lognormal(mean=mu(m_val, m_spread), sigma=m_spread, size=num_nodes)
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = rng.lognormal(mean=mu(c_val, c_spread), sigma=c_spread, size=num_nodes)
    c_diag = np.repeat(c_nodes, dims)

    k_connection_vals = rng.lognormal(
        mean=mu(k_wall_val, k_wall_spread), sigma=k_wall_spread, size=connection_src.shape[0] // 2
    ).repeat(2)
    k_between_vals = rng.lognormal(
        mean=mu(k_between_val, k_between_spread), sigma=k_between_spread, size=between_src.shape[0]
    )
    k_vals = np.concatenate([k_connection_vals, k_between_vals])

    rest_connection_lens = rng.lognormal(
        mean=mu(wall_rest_val, wall_rest_spread),
        sigma=wall_rest_spread,
        size=connection_src.shape[0] // 2,
    ).repeat(2)
    rest_between_lens = rng.lognormal(
        mean=mu(between_rest_val, between_rest_spread), sigma=between_rest_spread, size=between_src.shape[0]
    )
    rest_lens = np.concatenate([rest_connection_lens, rest_between_lens])

    displacement, velocity = run_simulation(
        steps=total_steps,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        rest_lens=rest_lens,
        wall_nodes=wall_nodes,
    )

    movement_nodes = 1 + np.arange(N) * 3
    movement_idx = movement_nodes * dims
    X = np.column_stack((displacement[:, movement_idx], velocity[:, movement_idx]))

    return X, u

In [23]:
def memory_trial(
    input, X_states, k_max, total_steps, start_idx, train_steps, test_steps, ridge_alpha
):
    u = input
    X = X_states

    X_clean = X[start_idx:]
    X_train = X_clean[:train_steps]
    X_test = X_clean[train_steps : train_steps + test_steps]

    Y_deg1 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg1[k:, k - 1] = np.sqrt(3.0) * u_delayed
    Y_deg2 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg2[k:, k - 1] = np.sqrt(5.0) * (3.0 * (u_delayed**2) - 1.0) / 2.0

    Y_deg3 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg3[k:, k - 1] = (
            np.sqrt(7.0) * (5.0 * (u_delayed**3) - 3.0 * u_delayed) / 2.0
        )

    # Degree 1
    Y1_clean = Y_deg1[start_idx:]
    Y1_train = Y1_clean[:train_steps]
    Y1_test = Y1_clean[train_steps : train_steps + test_steps]
    c_deg1 = evaluate_capacity(X_train, X_test, Y1_train, Y1_test, k_max, ridge_alpha)
    # Degree 2
    Y2_clean = Y_deg2[start_idx:]
    Y2_train = Y2_clean[:train_steps]
    Y2_test = Y2_clean[train_steps : train_steps + test_steps]
    c_deg2 = evaluate_capacity(X_train, X_test, Y2_train, Y2_test, k_max, ridge_alpha)
    # Degree 3
    Y3_clean = Y_deg3[start_idx:]
    Y3_train = Y3_clean[:train_steps]
    Y3_test = Y3_clean[train_steps : train_steps + test_steps]
    c_deg3 = evaluate_capacity(X_train, X_test, Y3_train, Y3_test, k_max, ridge_alpha)

    k_cross = 25
    cross_pairs = [
        (k1, k2) for k1 in range(1, k_cross + 1) for k2 in range(k1 + 1, k_cross + 1)
    ]
    num_cross_pairs = len(cross_pairs)
    Y_cross2 = np.zeros((total_steps, num_cross_pairs))
    for idx, (k1, k2) in enumerate(cross_pairs):
        max_k = k2
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]
        Y_cross2[max_k:, idx] = 3.0 * (u_k1 * u_k2)

    # Cross Degree 2
    Y_cross2_clean = Y_cross2[start_idx:]
    c_cross2 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross2_clean[:train_steps],
        Y_cross2_clean[train_steps : train_steps + test_steps],
        num_cross_pairs,
        ridge_alpha,
    )

    k_cross3 = 25
    # Cross Degree 3 (2+1 interaction)
    pairs_21 = [
        (k1, k2)
        for k1 in range(1, k_cross3 + 1)
        for k2 in range(1, k_cross3 + 1)
        if k1 != k2
    ]
    num_21 = len(pairs_21)
    Y_cross3_21 = np.zeros((total_steps, num_21))
    for idx, (k1, k2) in enumerate(pairs_21):
        max_k = max(k1, k2)
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]

        # sqrt(5) * P2 * sqrt(3) * P1 = sqrt(15) * ((3*u^2 - 1)/2) * u
        p2 = np.sqrt(5.0) * (3.0 * (u_k1**2) - 1.0) / 2.0
        p1 = np.sqrt(3.0) * u_k2
        Y_cross3_21[max_k:, idx] = p2 * p1
    # Cross Degree 3 (1+1+1 interaction)
    triples_111 = [
        (k1, k2, k3)
        for k1 in range(1, k_cross3 + 1)
        for k2 in range(k1 + 1, k_cross3 + 1)
        for k3 in range(k2 + 1, k_cross3 + 1)
    ]
    num_111 = len(triples_111)
    Y_cross3_111 = np.zeros((total_steps, num_111))
    for idx, (k1, k2, k3) in enumerate(triples_111):
        max_k = k3
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]
        u_k3 = u[max_k - k3 : len(u) - k3, 0]

        Y_cross3_111[max_k:, idx] = (3.0 * np.sqrt(3.0)) * (u_k1 * u_k2 * u_k3)
    # Cross Degree 3 (2+1 interaction)
    c_cross3_21 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross3_21[start_idx : start_idx + train_steps],
        Y_cross3_21[start_idx + train_steps : start_idx + train_steps + test_steps],
        num_21,
        ridge_alpha,
    )
    # Cross Degree 3 (1+1+1 interaction)
    c_cross3_111 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross3_111[start_idx : start_idx + train_steps],
        Y_cross3_111[start_idx + train_steps : start_idx + train_steps + test_steps],
        num_111,
        ridge_alpha,
    )

    return [
        c_deg1,
        c_deg2,
        c_deg3,
        c_cross2,
        c_cross3_21,
        c_cross3_111,
    ]

In [24]:
rng_vals = [0, 42, 123, 456, 789, 999]

rng_trials_results_avg = np.zeros(6)

for i, rng_val in enumerate(rng_vals):
    sigma = 0.5
    X, u = spring_trial(
        rng_seed=rng_val,
        input_force=10,
        m_val=0.01,
        m_spread=sigma,
        c_val=0.2,
        c_spread=sigma,
        k_wall_val=5,
        k_wall_spread=sigma,
        k_between_val=4,
        k_between_spread=sigma,
        wall_rest_val=dist_y,
        wall_rest_spread=sigma,
        between_rest_val=dist_between,
        between_rest_spread=sigma,
    )

    values = memory_trial(
        u,
        X,
        k_max,
        total_steps,
        start_idx,
        train_steps,
        test_steps,
        ridge_alpha=1e-5,
    )
    values_sum = np.array([np.sum(c) for c in values])
    rng_trials_results_avg += values_sum

rng_trials_results_avg = rng_trials_results_avg / len(rng_vals)

In [25]:
categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

fig_bar = go.Figure(
    data=go.Bar(
        x=categories,
        y=rng_trials_results_avg,
        text=[f"{v:.2f}" for v in rng_trials_results_avg],
        textposition="auto",
    )
)

fig_bar.update_layout(
    title=f"Total Capacity: {sum(rng_trials_results_avg):.2f}",
    xaxis_title="Capacity Component",
    yaxis_title="Total Capacity (MC)",
    template="plotly_white",
    width=750,
    height=450,
)

fig_bar.show()

In [26]:
rng = np.random.default_rng(42)


def objective(trial):
    if trial.number % 10 == 0:
        print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)
    trial_seeds = rng.integers(0, 2**31 - 1, size=3)
    trial.set_user_attr("trial_seeds", trial_seeds)

    start_time = time.perf_counter()
    rng_trials_results_avg = np.zeros(6)
    for trial_seed in trial_seeds:
        X, u = spring_trial(
            rng_seed=trial_seed,
            input_force=trial.suggest_float("input_force", 1, 50),
            m_val=trial.suggest_float("m_val", 0.001, 0.1, log=True),
            m_spread=trial.suggest_float("m_spread", 0, 1.5, log=False),
            c_val=trial.suggest_float("c_val", 0.01, 1.0, log=True),
            c_spread=trial.suggest_float("c_spread", 0, 1.5, log=False),
            k_wall_val=trial.suggest_float("k_wall_val", 1, 100, log=True),
            k_wall_spread=trial.suggest_float("k_wall_spread", 0, 1.5, log=False),
            k_between_val=trial.suggest_float("k_between_val", 1, 100, log=True),
            k_between_spread=trial.suggest_float("k_between_spread", 0, 1.5, log=False),
            wall_rest_val=dist_y,
            wall_rest_spread=trial.suggest_float("wall_rest_spread", 0, 1.5, log=False),
            between_rest_val=dist_between,
            between_rest_spread=trial.suggest_float(
                "between_rest_spread", 0, 1.5, log=False
            ),
        )
        if np.isnan(X).any():
            raise optuna.TrialPruned("NaN detected in spring_trial results.")
        values = memory_trial(
            u,
            X,
            k_max,
            total_steps,
            start_idx,
            train_steps,
            test_steps,
            ridge_alpha=trial.suggest_float("ridge_alpha", 1e-6, 1e2, log=True),
        )
        values_sum = np.array([np.sum(c) for c in values])
        rng_trials_results_avg += values_sum
    elapsed_time = time.perf_counter() - start_time
    trial.set_user_attr("duration_sec", elapsed_time)

    results = rng_trials_results_avg / len(trial_seeds)
    return results[0], results[1], results[2], results[3], results[4], results[5]


study = optuna.create_study(directions=["maximize"] * 6)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, timeout=60, n_jobs=-1)  # timeout, n_trials

[Optuna] Processing Trial #0...

[Optuna] Processing Trial #120...

In [27]:
pruned_trials = [t for t in study.trials if t.state == TrialState.PRUNED]
failed_trials = [t for t in study.trials if t.state == TrialState.FAIL]
complete_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

print(f"Total Trials:    {len(study.trials)}")
print(f"Completed:       {len(complete_trials)}")
print(f"Pruned (NaNs):   {len(pruned_trials)}")
print(f"Failed (Crashes): {len(failed_trials)}")

Total Trials:    130
Completed:       38
Pruned (NaNs):   92
Failed (Crashes): 0


In [28]:
for trial in study.best_trials[:3]:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #4
  Values: [16.68782456432955, 0.10383414140748609, 0.1274132056699138, 0.29359194573586683, 0.1869010274368662, 0.7090330449152474]
  Params: {'input_force': 21.833513326377503, 'm_val': 0.0826674542563098, 'm_spread': 0.2999238485695065, 'c_val': 0.32511712373461193, 'c_spread': 1.4721101371515717, 'k_wall_val': 37.60487431416067, 'k_wall_spread': 0.2347497358330533, 'k_between_val': 22.694704877283062, 'k_between_spread': 0.2707535298695122, 'wall_rest_spread': 0.13495038837170376, 'between_rest_spread': 0.328091671933387, 'ridge_alpha': 4.643108996531821e-06}
Trial #21
  Values: [16.686278810161507, 0.3757304276497963, 0.06243627015229521, 4.3614706940200865, 0.24725536376793347, 1.0597510259451959]
  Params: {'input_force': 3.7344139927101807, 'm_val': 0.028328183898239778, 'm_spread': 1.3061146648899147, 'c_val': 0.20852749423615394, 'c_spread': 0.023613768461012707, 'k_wall_val': 4.311873726810907, 'k_wall_spread': 0.2080036832768959, 'k_between_val': 2.7158202137774223,

In [29]:
top_trials = sorted(study.best_trials, key=lambda t: sum(t.values), reverse=True)
n_trials = len(top_trials)

categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

cols = 2
rows = (n_trials + cols - 1) // cols

subplot_titles = [f"Trial #{t.number} | Total: {sum(t.values):.2f}" for t in top_trials]

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=subplot_titles,
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

for idx, trial in enumerate(top_trials):
    r = (idx // cols) + 1
    c = (idx % cols) + 1
    values = trial.values

    fig.add_trace(
        go.Bar(
            x=categories,
            y=values,
            text=[f"{v:.2f}" for v in values],
            textposition="auto",
            showlegend=False,
        ),
        row=r,
        col=c,
    )

    fig.update_xaxes(tickangle=-30, row=r, col=c)
    fig.update_yaxes(title_text="MC", row=r, col=c)

fig.update_layout(
    title_text="Top Optuna Trials - Memory Capacity Profiles (Sorted by Total MC)",
    template="plotly_white",
    height=320 * rows,
    width=1100,
)

fig.show()

In [30]:
rng = np.random.default_rng(42)


def objective(trial):
    if trial.number % 10 == 0:
        print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)
    trial_seeds = rng.integers(0, 2**31 - 1, size=3)
    trial.set_user_attr("trial_seeds", trial_seeds)

    start_time = time.perf_counter()
    rng_trials_results_avg = np.zeros(6)
    for trial_seed in trial_seeds:
        X, u = spring_trial(
            rng_seed=trial_seed,
            input_force=trial.suggest_float("input_force", 1, 50),
            m_val=trial.suggest_float("m_val", 0.001, 0.1, log=True),
            m_spread=0,
            c_val=trial.suggest_float("c_val", 0.01, 1.0, log=True),
            c_spread=0,
            k_wall_val=trial.suggest_float("k_wall_val", 1, 100, log=True),
            k_wall_spread=0,
            k_between_val=trial.suggest_float("k_between_val", 1, 100, log=True),
            k_between_spread=0,
            wall_rest_val=dist_y * 1.1,
            wall_rest_spread=0,
            between_rest_val=dist_between,
            between_rest_spread=0,
        )
        if np.isnan(X).any():
            raise optuna.TrialPruned("NaN detected in spring_trial results.")
        values = memory_trial(
            u,
            X,
            k_max,
            total_steps,
            start_idx,
            train_steps,
            test_steps,
            ridge_alpha=trial.suggest_float("ridge_alpha", 1e-6, 1e2, log=True),
        )
        values_sum = np.array([np.sum(c) for c in values])
        rng_trials_results_avg += values_sum
    elapsed_time = time.perf_counter() - start_time
    trial.set_user_attr("duration_sec", elapsed_time)

    results = rng_trials_results_avg / len(trial_seeds)
    return results[0], results[1], results[2], results[3], results[4], results[5]


order_study = optuna.create_study(directions=["maximize"] * 6)
optuna.logging.set_verbosity(optuna.logging.WARNING)
order_study.optimize(objective, timeout=60, n_jobs=-1)  # timeout, n_trials

[Optuna] Processing Trial #40...

In [31]:
pruned_trials = [t for t in order_study.trials if t.state == TrialState.PRUNED]
failed_trials = [t for t in order_study.trials if t.state == TrialState.FAIL]
complete_trials = [t for t in order_study.trials if t.state == TrialState.COMPLETE]

print(f"Total Trials:    {len(order_study.trials)}")
print(f"Completed:       {len(complete_trials)}")
print(f"Pruned (NaNs):   {len(pruned_trials)}")
print(f"Failed (Crashes): {len(failed_trials)}")

Total Trials:    48
Completed:       40
Pruned (NaNs):   8
Failed (Crashes): 0


In [32]:
top_trials = sorted(order_study.best_trials, key=lambda t: sum(t.values), reverse=True)
n_trials = len(top_trials)

categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

cols = 3
rows = (n_trials + cols - 1) // cols

subplot_titles = [f"Trial #{t.number} | Total: {sum(t.values):.2f}" for t in top_trials]

vert_spacing = (0.6 / (rows - 1)) if rows > 1 else 0.0

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=subplot_titles,
    vertical_spacing=vert_spacing,
    horizontal_spacing=0.06,
)

for idx, trial in enumerate(top_trials):
    r = (idx // cols) + 1
    c = (idx % cols) + 1
    values = trial.values

    fig.add_trace(
        go.Bar(
            x=categories,
            y=values,
            text=[f"{v:.2f}" for v in values],
            textposition="auto",
            showlegend=False,
        ),
        row=r,
        col=c,
    )

    fig.update_xaxes(tickangle=-30, row=r, col=c)
    fig.update_yaxes(title_text="MC", row=r, col=c)

fig.update_layout(
    title_text="Top Optuna Trials - Memory Capacity Profiles (Sorted by Total MC)",
    template="plotly_white",
    height=280 * rows,
    width=1200,
)

fig.show()

# Non Lin vs Lin

In [9]:
rng = np.random.default_rng(42)


def objective(trial):
    if trial.number % 10 == 0:
        print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)
    trial_seeds = rng.integers(0, 2**31 - 1, size=3)
    trial.set_user_attr("trial_seeds", trial_seeds)

    start_time = time.perf_counter()
    rng_trials_results_avg = np.zeros(6)
    for trial_seed in trial_seeds:
        X, u = spring_trial(
            rng_seed=trial_seed,
            input_force=trial.suggest_float("input_force", 1, 50),
            m_val=trial.suggest_float("m_val", 0.001, 0.1, log=True),
            m_spread=0,
            c_val=trial.suggest_float("c_val", 0.01, 1.0, log=True),
            c_spread=0,
            k_wall_val=trial.suggest_float("k_wall_val", 1, 100, log=True),
            k_wall_spread=0,
            k_between_val=trial.suggest_float("k_between_val", 1, 100, log=True),
            k_between_spread=0,
            wall_rest_val=dist_y * 1.1,
            wall_rest_spread=0,
            between_rest_val=dist_between,
            between_rest_spread=0,
        )
        if np.isnan(X).any():
            raise optuna.TrialPruned("NaN detected in spring_trial results.")
        values = memory_trial(
            u,
            X,
            k_max,
            total_steps,
            start_idx,
            train_steps,
            test_steps,
            ridge_alpha=trial.suggest_float("ridge_alpha", 1e-6, 1e2, log=True),
        )
        values_sum = np.array([np.sum(c) for c in values])
        rng_trials_results_avg += values_sum
    elapsed_time = time.perf_counter() - start_time
    trial.set_user_attr("duration_sec", elapsed_time)
    
    results = rng_trials_results_avg / len(trial_seeds)
    linear_cap = results[0]
    nonlinear_cap = results[1] + results[2] + results[3] + results[4] + results[5]
    return linear_cap, nonlinear_cap


order_study = optuna.create_study(directions=["maximize"] * 2)
optuna.logging.set_verbosity(optuna.logging.WARNING)
order_study.optimize(objective, timeout=60, n_jobs=-1)  # timeout, n_trials

[I 2026-10-05 18:34:24,293] A new study created in memory with name: no-name-0fc88727-69c9-4775-8052-7dc8322c5983


[Optuna] Processing Trial #0...

[W 2026-10-05 18:34:24,296] Trial 1 failed with parameters: {} because of the following error: NameError("name 'spring_trial' is not defined").
Traceback (most recent call last):
  File "/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_6748/2618614366.py", line 13, in objective
    X, u = spring_trial(
           ^^^^^^^^^^^^
NameError: name 'spring_trial' is not defined
[W 2026-10-05 18:34:24,297] Trial 2 failed with parameters: {} because of the following error: NameError("name 'spring_trial' is not defined").
Traceback (most recent call last):
  File "/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_6748/2618614366.

NameError: name 'spring_trial' is not defined

In [46]:
pruned_trials = [t for t in order_study.trials if t.state == TrialState.PRUNED]
failed_trials = [t for t in order_study.trials if t.state == TrialState.FAIL]
complete_trials = [t for t in order_study.trials if t.state == TrialState.COMPLETE]

print(f"Total Trials:    {len(order_study.trials)}")
print(f"Completed:       {len(complete_trials)}")
print(f"Pruned (NaNs):   {len(pruned_trials)}")
print(f"Failed (Crashes): {len(failed_trials)}")

Total Trials:    36
Completed:       32
Pruned (NaNs):   4
Failed (Crashes): 0


In [47]:
top_trials = sorted(order_study.best_trials, key=lambda t: sum(t.values), reverse=True)
n_trials = len(top_trials)

categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

cols = 3
rows = (n_trials + cols - 1) // cols

subplot_titles = [f"Trial #{t.number} | Total: {sum(t.values):.2f}" for t in top_trials]

vert_spacing = (0.6 / (rows - 1)) if rows > 1 else 0.0

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=subplot_titles,
    vertical_spacing=vert_spacing,
    horizontal_spacing=0.06,
)

for idx, trial in enumerate(top_trials):
    r = (idx // cols) + 1
    c = (idx % cols) + 1
    values = trial.values

    fig.add_trace(
        go.Bar(
            x=categories,
            y=values,
            text=[f"{v:.2f}" for v in values],
            textposition="auto",
            showlegend=False,
        ),
        row=r,
        col=c,
    )

    fig.update_xaxes(tickangle=-30, row=r, col=c)
    fig.update_yaxes(title_text="MC", row=r, col=c)

fig.update_layout(
    title_text="Top Optuna Trials - Memory Capacity Profiles (Sorted by Total MC)",
    template="plotly_white",
    height=280 * rows,
    width=1200,
)

fig.show()

In [48]:
rng = np.random.default_rng(42)


def objective(trial):
    if trial.number % 10 == 0:
        print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)
    trial_seeds = rng.integers(0, 2**31 - 1, size=3)
    trial.set_user_attr("trial_seeds", trial_seeds)

    start_time = time.perf_counter()
    rng_trials_results_avg = np.zeros(6)
    for trial_seed in trial_seeds:
        X, u = spring_trial(
            rng_seed=trial_seed,
            input_force=trial.suggest_float("input_force", 1, 50),
            m_val=trial.suggest_float("m_val", 0.001, 0.1, log=True),
            m_spread=trial.suggest_float("m_spread", 0, 1.5, log=False),
            c_val=trial.suggest_float("c_val", 0.01, 1.0, log=True),
            c_spread=trial.suggest_float("c_spread", 0, 1.5, log=False),
            k_wall_val=trial.suggest_float("k_wall_val", 1, 100, log=True),
            k_wall_spread=trial.suggest_float("k_wall_spread", 0, 1.5, log=False),
            k_between_val=trial.suggest_float("k_between_val", 1, 100, log=True),
            k_between_spread=trial.suggest_float("k_between_spread", 0, 1.5, log=False),
            wall_rest_val=dist_y,
            wall_rest_spread=trial.suggest_float("wall_rest_spread", 0, 1.5, log=False),
            between_rest_val=dist_between,
            between_rest_spread=trial.suggest_float(
                "between_rest_spread", 0, 1.5, log=False
            ),
        )
        if np.isnan(X).any():
            raise optuna.TrialPruned("NaN detected in spring_trial results.")
        values = memory_trial(
            u,
            X,
            k_max,
            total_steps,
            start_idx,
            train_steps,
            test_steps,
            ridge_alpha=trial.suggest_float("ridge_alpha", 1e-6, 1e2, log=True),
        )
        values_sum = np.array([np.sum(c) for c in values])
        rng_trials_results_avg += values_sum
    elapsed_time = time.perf_counter() - start_time
    trial.set_user_attr("duration_sec", elapsed_time)

    results = rng_trials_results_avg / len(trial_seeds)
    return results[0] + results[1] + results[2], results[3] + results[4] + results[5]


disorder_study = optuna.create_study(directions=["maximize"] * 2)
optuna.logging.set_verbosity(optuna.logging.WARNING)
disorder_study.optimize(objective, timeout=60, n_jobs=-1)  # timeout, n_trials

[Optuna] Processing Trial #120...

In [49]:
pruned_trials = [t for t in disorder_study.trials if t.state == TrialState.PRUNED]
failed_trials = [t for t in disorder_study.trials if t.state == TrialState.FAIL]
complete_trials = [t for t in disorder_study.trials if t.state == TrialState.COMPLETE]

print(f"Total Trials:    {len(disorder_study.trials)}")
print(f"Completed:       {len(complete_trials)}")
print(f"Pruned (NaNs):   {len(pruned_trials)}")
print(f"Failed (Crashes): {len(failed_trials)}")

Total Trials:    125
Completed:       35
Pruned (NaNs):   90
Failed (Crashes): 0


In [50]:
top_trials = sorted(disorder_study.best_trials, key=lambda t: sum(t.values), reverse=True)
n_trials = len(top_trials)

categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

cols = 2
rows = (n_trials + cols - 1) // cols

subplot_titles = [f"Trial #{t.number} | Total: {sum(t.values):.2f}" for t in top_trials]

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=subplot_titles,
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

for idx, trial in enumerate(top_trials):
    r = (idx // cols) + 1
    c = (idx % cols) + 1
    values = trial.values

    fig.add_trace(
        go.Bar(
            x=categories,
            y=values,
            text=[f"{v:.2f}" for v in values],
            textposition="auto",
            showlegend=False,
        ),
        row=r,
        col=c,
    )

    fig.update_xaxes(tickangle=-30, row=r, col=c)
    fig.update_yaxes(title_text="MC", row=r, col=c)

fig.update_layout(
    title_text="Top Optuna Trials - Memory Capacity Profiles (Sorted by Total MC)",
    template="plotly_white",
    height=320 * rows,
    width=1100,
)

fig.show()

In [ ]:
fig = plot_pareto_front(
    disorder_study,
    target_names=["Linear Capacity", "Nonlinear Capacity"],
    include_dominated_trials=True,
)

fig.update_layout(
    title="Pareto Front: Linear vs. Nonlinear Memory Capacity",
    template="plotly_white",
    width=800,
    height=550,
)

fig.show()